In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

In [ ]:
# configure the data path
DRIVE_ROOT = '/content/drive/MyDrive/16th grade Academics/A S26/CSCI 6980'
DATA_SOURCE_PATH = os.path.join(DRIVE_ROOT, 'SHL')
PERMANENT_DATASET_PATH = os.path.join(DRIVE_ROOT, 'YOLO_HOCKEY_DATASET')

In [ ]:
def setup_permanent_dataset(overwrite=False):
    """
    Restructures SHL data into a permanent YOLO format on Google Drive.
    """
    # 1. Safety Check: Don't accidentally wipe your Drive!
    if os.path.exists(PERMANENT_DATASET_PATH):
        if overwrite:
            print(f"Overwriting existing dataset at: {PERMANENT_DATASET_PATH}")
            shutil.rmtree(PERMANENT_DATASET_PATH)
        else:
            print(f"Dataset already exists at {PERMANENT_DATASET_PATH}. Skipping.")
            return

    # 2. Create the YOLO directory structure
    subdirs = [
        'images/train', 'images/val',
        'labels/train', 'labels/val'
    ]
    for d in subdirs:
        os.makedirs(os.path.join(PERMANENT_DATASET_PATH, d), exist_ok=True)

    # 3. Define Source Paths
    frames_dir = os.path.join(DATA_SOURCE_PATH, 'frames')
    anns_dir = os.path.join(DATA_SOURCE_PATH, 'annotations')

    if not os.path.exists(frames_dir):
        raise FileNotFoundError(f"Could not find frames at {frames_dir}")

    # 4. Get and Split Files
    images = [f for f in os.listdir(frames_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

    print(f"Copying {len(train_imgs)} train and {len(val_imgs)} val files to permanent Drive folder...")

    def copy_batch(file_list, split_name):
        for img_name in file_list:
            # Copy Image
            src_img = os.path.join(frames_dir, img_name)
            dst_img = os.path.join(PERMANENT_DATASET_PATH, 'images', split_name, img_name)
            shutil.copy2(src_img, dst_img) # copy2 preserves metadata

            # Copy Annotation
            txt_name = os.path.splitext(img_name)[0] + '.txt'
            src_txt = os.path.join(anns_dir, txt_name)
            if os.path.exists(src_txt):
                dst_txt = os.path.join(PERMANENT_DATASET_PATH, 'labels', split_name, txt_name)
                shutil.copy2(src_txt, dst_txt)

    # 5. Execute
    copy_batch(train_imgs, 'train')
    copy_batch(val_imgs, 'val')
    print(f"Success! Dataset is now permanent at: {PERMANENT_DATASET_PATH}")

# Run the function
setup_permanent_dataset(overwrite=False)

Copying 1680 train and 421 val files to permanent Drive folder...
Success! Dataset is now permanent at: /content/drive/MyDrive/16th grade Academics/A S26/CSCI 6980/YOLO_HOCKEY_DATASET


In [ ]:
import yaml
import os

# Define the permanent path where your dataset lives
PERMANENT_ROOT = '/content/drive/MyDrive/16th grade Academics/A S26/CSCI 6980/YOLO_HOCKEY_DATASET'

data_config = {
    'path': PERMANENT_ROOT,    # The absolute root of the dataset
    'train': 'images/train',   # Path relative to 'path'
    'val': 'images/val',       # Path relative to 'path'
    'names': {
        0: 'centerIce',
        1: 'faceoff',
        2: 'goal',
        3: 'goaltender',
        4: 'player',
        5: 'puck',
        6: 'referee'
    }
}

# Save the YAML file directly into your permanent Drive folder
yaml_file_path = os.path.join(PERMANENT_ROOT, 'hockey_data.yaml')

with open(yaml_file_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"Permanent YAML config saved to: {yaml_file_path}")

Permanent YAML config saved to: /content/drive/MyDrive/16th grade Academics/A S26/CSCI 6980/YOLO_HOCKEY_DATASET/hockey_data.yaml
